In [40]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [45]:
df = pd.read_csv("douro_vineyard_data.csv")
# descriptive statistics
df.describe()

,soil_depth_cm,sun_hours_july,vine_density_per_ha,annual_rainfall_mm,distance_from_porto_km,elevation_m,soil_ph,harvest_success
count,3543.000000,3543.000000,3543.000000,3543.000000,3543.000000,3543.000000,3543.000000,3543.000000
mean,98.703076,340.410387,4506.006774,651.746542,131.025713,221.822636,6.299083,0.291279
std,41.715072,24.558836,606.159716,120.830834,40.796542,69.793943,0.342307,0.454416
min,40.000000,250.000000,2500.000000,250.000000,70.000000,40.000000,5.190000,0.000000
25%,56.000000,324.000000,4087.500000,570.000000,92.200000,174.200000,6.070000,0.000000
50%,114.000000,340.000000,4512.000000,654.000000,131.100000,221.600000,6.310000,0.000000
75%,138.000000,357.000000,4918.000000,732.000000,165.500000,269.800000,6.530000,1.000000
max,160.000000,441.000000,6637.000000,1040.000000,204.900000,451.600000,7.580000,1.000000


In [46]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3543 entries, 0 to 3542
Data columns (total 17 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   plot_id                 3543 non-null   object 
 1   treatment               3543 non-null   object 
 2   soil_depth_cm           3543 non-null   float64
 3   trellis_system          3543 non-null   object 
 4   vineyard_age_group      3543 non-null   object 
 5   sun_hours_july          3543 non-null   float64
 6   vine_density_per_ha     3543 non-null   float64
 7   cooperative             3543 non-null   object 
 8   annual_rainfall_mm      3543 non-null   float64
 9   grape_variety           3543 non-null   object 
 10  distance_from_porto_km  3543 non-null   float64
 11  organic_certified       3543 non-null   object 
 12  elevation_m             3543 non-null   float64
 13  soil_ph                 3543 non-null   float64
 14  harvest_month           3543 non-null   

In [49]:
# df.groupby('trellis_system')['harvest_success'].value_counts()
df.groupby(['trellis_system', 'treatment']).agg({'harvest_success': 'mean'})

harvest_success
trellis_system             treatment                 
bush vine                  control           0.226636
                           treatment         0.297917
lyre                       control           0.268371
                           treatment         0.326870
vertical shoot positioning control           0.291667
                           treatment         0.308866

We search for a two-group split on each column. Categorical variables like trellis_system, harvest_month, age_group are analysed separately. We are looking specifically for a variable where the relationship average harvest_success for treatment is lower than the average harvest_success for control at the split or cutoff (for numerical variables). Below the code that considers both categorical and numerical approach.

In [29]:
# Search categorical groups and all possible binary splits of numeric features

print("Overall rates:")
print(df.groupby("treatment")["harvest_success"].mean())

candidates = [col for col in df.columns if col not in {"plot_id", "treatment", "harvest_success"}]
found = []
for col in candidates:
    if df[col].dtype == "object":
        grouped = df.groupby([col, "treatment"])["harvest_success"].agg(["mean", "size"]).unstack("treatment")
        grouped = grouped.dropna(subset=[("mean", "control"), ("mean", "treatment")])
        if len(grouped) > 0 and (grouped[("mean", "treatment")] < grouped[("mean", "control")]).all():
            found.append((col, "categorical", grouped))
    else:
        data = df[[col, "treatment", "harvest_success"]].dropna().sort_values(col)
        vals = data[col].unique()
        for threshold in vals[:-1]:
            low = data[data[col] <= threshold]
            high = data[data[col] > threshold]
            rates = pd.concat({
                f"<= {threshold}": low.groupby("treatment")["harvest_success"].mean(),
                f"> {threshold}": high.groupby("treatment")["harvest_success"].mean(),
            }, axis=1).T
            if {"control", "treatment"}.issubset(rates.columns) and (rates["treatment"] < rates["control"]).all():
                found.append((col, f"binary threshold {threshold}", rates))
                break

for col, grouping, rates in found:
    print(f"\nCOMPLETE REVERSAL: {col} ({grouping})")
    print(rates)
print("\nFound:", [col for col, _, _ in found])

Overall rates:
treatment
control      0.270417
treatment    0.309524
Name: harvest_success, dtype: float64

COMPLETE REVERSAL: distance_from_porto_km (binary threshold 102.9)
treatment   control  treatment
<= 102.9   0.252508   0.239437
> 102.9    0.317287   0.315217

Found: ['distance_from_porto_km']
